## Predicción de los costos para 3 años

Como se mostró en el análisis exploratorio, se optó por hacer el forecasting sobre los datos históricos de los equipos directamente. En este notebook se presentan los hallazgos con diferentes aproximaciones al problema y las estimaciones finales sugeridas, considerando 2025, 2026 y 2027 como los años de ejecución del proyecto.

Se decide utilizar Prophet como framework para realizar el forecasting, entre otros varios modelos disponibles en la literatura, debido a su facilidad de uso y robustez ante datos faltantes y outliers, lo que lo hace una herramienta idónea como primera aproximación al problema. Prophet permite descomponer la serie temporal en componentes interpretables (tendencia, estacionalidad) y generar intervalos de confianza para cuantificar la incertidumbre en las estimaciones.

In [74]:
import pandas as pd
import plotly.graph_objects as go
from prophet import Prophet

In [2]:
df = pd.read_csv('../data/processed/equipment_costs.csv')

df['Date'] = pd.to_datetime(df['Date'])

print("Equipment Costs Dataset")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Number of records: {len(df)}")
print(f"\nFirst rows:")
print(df.head())
print(f"\nDataset info:")
print(df.info())

Equipment Costs Dataset
Date range: 2010-01-04 00:00:00 to 2023-08-31 00:00:00
Number of records: 3530

First rows:
        Date  Price_X  Price_Y  Price_Z  Equipment_1  Equipment_2
0 2010-01-04    80.12    527.5  2225.25      438.024   944.290000
1 2010-01-05    80.59    527.5  2246.50      438.118   951.530000
2 2010-01-06    81.89    527.5  2302.50      438.378   970.630000
3 2010-01-07    81.51    527.5  2306.50      438.302   971.836667
4 2010-01-08    81.37    552.5  2261.25      458.274   965.040000

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3530 entries, 0 to 3529
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         3530 non-null   datetime64[ns]
 1   Price_X      3530 non-null   float64       
 2   Price_Y      3530 non-null   float64       
 3   Price_Z      3530 non-null   float64       
 4   Equipment_1  3530 non-null   float64       
 5   Equipment_2  3530 n

In [87]:
def train_and_forecast_equipment(df, equipment_col, train_end_date='2023-07-31', forecast_end_date='2027-12-31'):
    """
    Train Prophet model and generate forecasts for an equipment.

    Parameters:
    - df: DataFrame with Date and equipment price columns
    - equipment_col: Name of the equipment column to forecast
    - train_end_date: Last date to use for training (default: '2023-07-31')
    - forecast_end_date: End date for predictions (default: '2027-12-31')

    Returns:
    - model: Trained Prophet model
    - forecast: DataFrame with predictions
    """
    df_filtered = df[df['Date'] <= train_end_date].copy()

    df_prophet = df_filtered[['Date', equipment_col]].copy()
    df_prophet.columns = ['ds', 'y']

    print(f"\n{equipment_col} - Preparing model...")
    print(f"Training records: {len(df_prophet)}")
    print(f"Training date range: {df_prophet['ds'].min()} to {df_prophet['ds'].max()}")
    print(f"Price range: ${df_prophet['y'].min():.2f} to ${df_prophet['y'].max():.2f}")

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.90
    )

    model.fit(df_prophet)
    print(f"{equipment_col} - Model trained successfully")

    future = model.make_future_dataframe(
        periods=365*10,
        freq='D'
    )

    future = future[future['ds'] <= forecast_end_date]
    forecast = model.predict(future)

    print(f"{equipment_col} - Predictions made until {forecast['ds'].max()}")
    print(f"Total predictions: {len(forecast)}")

    return model, forecast

model_eq1, forecast_eq1 = train_and_forecast_equipment(df, 'Equipment_1', train_end_date='2023-08-31')

model_eq2, forecast_eq2 = train_and_forecast_equipment(df, 'Equipment_2', train_end_date='2023-08-31')

16:45:09 - cmdstanpy - INFO - Chain [1] start processing



Equipment_1 - Preparing model...
Training records: 3530
Training date range: 2010-01-04 00:00:00 to 2023-08-31 00:00:00
Price range: $213.22 to $863.76


16:45:09 - cmdstanpy - INFO - Chain [1] done processing


Equipment_1 - Model trained successfully


16:45:09 - cmdstanpy - INFO - Chain [1] start processing


Equipment_1 - Predictions made until 2027-12-31 00:00:00
Total predictions: 5113

Equipment_2 - Preparing model...
Training records: 3530
Training date range: 2010-01-04 00:00:00 to 2023-08-31 00:00:00
Price range: $577.03 to $1668.05


16:45:10 - cmdstanpy - INFO - Chain [1] done processing


Equipment_2 - Model trained successfully
Equipment_2 - Predictions made until 2027-12-31 00:00:00
Total predictions: 5113


In [91]:
def plot_equipment_forecast(df, forecast, equipment_col, color):
    """
    Create interactive plot showing historical data, forecast and uncertainty interval.

    Parameters:
    - df: Original DataFrame with historical data
    - forecast: Prophet forecast DataFrame
    - equipment_col: Name of the equipment column
    - color: Color for the forecast line

    Returns:
    - fig: Plotly figure object
    """
    df_hist = df[['Date', equipment_col]].copy()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_hist['Date'],
        y=df_hist[equipment_col],
        mode='markers',
        name='Data Histórica',
        marker=dict(color='#2c3e50', size=4)
    ))

    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat'],
        mode='lines',
        name='Forecast',
        line=dict(color=color, width=2)
    ))

    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat_upper'],
        mode='lines',
        name='Cota superior (90%)',
        line=dict(width=0),
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat_lower'],
        mode='lines',
        name='Cota Inferior',
        line=dict(width=0),
        fillcolor=f'rgba{tuple(list(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + [0.2])}',
        fill='tonexty'
    ))

    equipment_col_esp = {
        'Equipment_1': 'Equipo 1',
        'Equipment_2': 'Equipo 2',
    }

    fig.update_layout(
        title=f'{equipment_col_esp[equipment_col]} - Predicción del costo con incertidumbre (Hasta Dic 2027)',
        xaxis_title='Fecha',
        yaxis_title='Costo',
        template='plotly_white',
        height=400,
        hovermode='x unified'
    )

    return fig

fig_eq1 = plot_equipment_forecast(df, forecast_eq1, 'Equipment_1', '#e74c3c')
fig_eq1.show()

fig_eq2 = plot_equipment_forecast(df, forecast_eq2, 'Equipment_2', '#9b59b6')
fig_eq2.show()

En la primera aproximación se utilizan todos los datos con el muestreo diario dispuesto inicialmente. Sin embargo, se identifica que para el horizonte de tiempo requerido, la incertidumbre de la predicción tiene un comportamiento casi exponencial, lo que reduce significativamente la confiabilidad de las estimaciones.

Para subsanar esto, se proponen las siguientes estrategias:
- **Utilizar un muestreo mensual de los datos**, agrupando los costos diarios mediante suma. Esto con el fin de disminuir el ruido en los datos, reducir la cantidad de puntos muestrales para el ajuste del modelo y estabilizar la predicción en el horizonte futuro.
- **Descartar los datos posteriores a diciembre 2020**, limitando el entrenamiento del modelo a datos hasta esta fecha para evitar periodos de alta volatilidad reciente que podrían sesgar las proyecciones a largo plazo.


In [93]:
def train_and_forecast_equipment_monthly(df, equipment_col, train_end_date='2020-12-31', forecast_end_date='2027-12-31'):
    """
    Train Prophet model using monthly resampled data and generate forecasts.

    Parameters:
    - df: DataFrame with Date and equipment price columns
    - equipment_col: Name of the equipment column to forecast
    - train_end_date: Last date to use for training (default: '2023-07-31')
    - forecast_end_date: End date for predictions (default: '2026-12-31')

    Returns:
    - model: Trained Prophet model
    - forecast: DataFrame with predictions
    - df_monthly: Monthly resampled data used for training
    """
    df_filtered = df[df['Date'] <= train_end_date].copy()

    df_monthly = df_filtered.set_index('Date')[equipment_col].resample('M').sum().reset_index()
    df_monthly.columns = ['ds', 'y']

    print(f"\n{equipment_col} - Preparing model with MONTHLY data (summed)...")
    print(f"Original daily records: {len(df_filtered)}")
    print(f"Monthly records after resampling: {len(df_monthly)}")
    print(f"Training date range: {df_monthly['ds'].min()} to {df_monthly['ds'].max()}")
    print(f"Monthly sum range: ${df_monthly['y'].min():.2f} to ${df_monthly['y'].max():.2f}")

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.90
    )

    model.fit(df_monthly)
    print(f"{equipment_col} - Model trained successfully")

    months_to_forecast = (pd.to_datetime(forecast_end_date) - df_monthly['ds'].max()).days // 30 + 12

    future = model.make_future_dataframe(
        periods=months_to_forecast,
        freq='ME'
    )

    future = future[future['ds'] <= forecast_end_date]

    forecast = model.predict(future)

    print(f"{equipment_col} - Monthly predictions made until {forecast['ds'].max()}")
    print(f"Total monthly predictions: {len(forecast)}")

    return model, forecast, df_monthly

model_eq1, forecast_eq1, monthly_eq1 = train_and_forecast_equipment_monthly(
    df, 'Equipment_1', train_end_date='2020-12-31'
)

model_eq2, forecast_eq2, monthly_eq2 = train_and_forecast_equipment_monthly(
    df, 'Equipment_2', train_end_date='2020-12-31'
)

/var/folders/9d/hm7x0fr139lfq6w19ymg575m0000gn/T/ipykernel_75607/1962229220.py:18: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

16:48:12 - cmdstanpy - INFO - Chain [1] start processing
16:48:12 - cmdstanpy - INFO - Chain [1] done processing
/var/folders/9d/hm7x0fr139lfq6w19ymg575m0000gn/T/ipykernel_75607/1962229220.py:18: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

16:48:12 - cmdstanpy - INFO - Chain [1] start processing
16:48:12 - cmdstanpy - INFO - Chain [1] done processing



Equipment_1 - Preparing model with MONTHLY data (summed)...
Original daily records: 2841
Monthly records after resampling: 132
Training date range: 2010-01-31 00:00:00 to 2020-12-31 00:00:00
Monthly sum range: $4471.70 to $13773.49
Equipment_1 - Model trained successfully
Equipment_1 - Monthly predictions made until 2027-12-31 00:00:00
Total monthly predictions: 216

Equipment_2 - Preparing model with MONTHLY data (summed)...
Original daily records: 2841
Monthly records after resampling: 132
Training date range: 2010-01-31 00:00:00 to 2020-12-31 00:00:00
Monthly sum range: $11883.42 to $25965.07
Equipment_2 - Model trained successfully
Equipment_2 - Monthly predictions made until 2027-12-31 00:00:00
Total monthly predictions: 216


In [94]:
def plot_equipment_forecast_monthly(df_monthly, forecast, equipment_col, color):
    """
    Create interactive plot showing historical monthly data, forecast and uncertainty interval.

    Parameters:
    - df_monthly: Monthly resampled DataFrame with historical data
    - forecast: Prophet forecast DataFrame
    - equipment_col: Name of the equipment column
    - color: Color for the forecast line

    Returns:
    - fig: Plotly figure object
    """
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_monthly['ds'],
        y=df_monthly['y'],
        mode='markers+lines',
        name='Data histórica (Mensual)',
        marker=dict(color='#2c3e50', size=6),
        line=dict(color='#2c3e50', width=1)
    ))

    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat'],
        mode='lines',
        name='Predicción',
        line=dict(color=color, width=2.5)
    ))

    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat_upper'],
        mode='lines',
        name='Cota superior (90%)',
        line=dict(width=0),
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=forecast['ds'],
        y=forecast['yhat_lower'],
        mode='lines',
        name='Cota inferior (90%)',
        line=dict(width=0),
        fillcolor=f'rgba{tuple(list(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + [0.2])}',
        fill='tonexty'
    ))

    fig.update_layout(
        title=f'{equipment_col} - Predicción del costo mensual (Hasta dic de 2027)',
        xaxis_title='Fecha',
        yaxis_title='Suma del precio mensual',
        template='plotly_white',
        height=400,
        hovermode='x unified'
    )

    return fig

fig_eq1 = plot_equipment_forecast_monthly(monthly_eq1, forecast_eq1, 'Equipo 1', '#e74c3c')
fig_eq1.show()

fig_eq2 = plot_equipment_forecast_monthly(monthly_eq2, forecast_eq2, 'Equipo 2', '#9b59b6')
fig_eq2.show()

Procedemos a estimar el costo de cada equipo para la duración del proyecto (36 meses)

In [85]:
# Cell 8: Calculate total costs for years 2025, 2026, 2027
def calculate_yearly_costs(forecast_df, years=[2025, 2026, 2027]):
    """
    Calculate yearly and total costs from Prophet forecast.

    Parameters:
    - forecast_df: Prophet forecast DataFrame
    - years: List of years to calculate costs for

    Returns:
    - Dictionary with yearly costs and total
    """
    results = {}

    # Calculate costs for each year
    for year in years:
        year_data = forecast_df[forecast_df['ds'].dt.year == year]
        results[year] = {
            'forecast': year_data['yhat'].sum(),
            'lower': year_data['yhat_lower'].sum(),
            'upper': year_data['yhat_upper'].sum()
        }

    # Calculate total across all years
    results['total'] = {
        'forecast': sum(results[year]['forecast'] for year in years),
        'lower': sum(results[year]['lower'] for year in years),
        'upper': sum(results[year]['upper'] for year in years)
    }

    return results

def print_cost_summary(costs, equipment_name):
    """
    Print formatted cost summary.

    Parameters:
    - costs: Dictionary from calculate_yearly_costs()
    - equipment_name: Name of the equipment
    """
    print("="*70)
    print(f"{equipment_name} - COST ESTIMATES")
    print("="*70)

    # Print yearly costs
    for year in [2025, 2026, 2027]:
        print(f"\n{year}:")
        print(f"  Forecast: ${costs[year]['forecast']:,.2f}")
        print(f"  Lower:    ${costs[year]['lower']:,.2f}")
        print(f"  Upper:    ${costs[year]['upper']:,.2f}")

    # Print total
    print(f"\n3-YEAR TOTAL (2025-2027):")
    print(f"  Forecast: ${costs['total']['forecast']:,.2f}")
    print(f"  Lower:    ${costs['total']['lower']:,.2f}")
    print(f"  Upper:    ${costs['total']['upper']:,.2f}")
    print()

# Calculate costs for both equipments
costs_eq1 = calculate_yearly_costs(forecast_eq1)
costs_eq2 = calculate_yearly_costs(forecast_eq2)

# Print summaries
print_cost_summary(costs_eq1, "EQUIPMENT 1")
print_cost_summary(costs_eq2, "EQUIPMENT 2")

EQUIPMENT 1 - COST ESTIMATES

2025:
  Forecast: $128,887.05
  Lower:    $101,019.47
  Upper:    $156,783.22

2026:
  Forecast: $132,746.51
  Lower:    $103,209.23
  Upper:    $161,762.14

2027:
  Forecast: $136,604.23
  Lower:    $105,496.57
  Upper:    $167,631.78

3-YEAR TOTAL (2025-2027):
  Forecast: $398,237.80
  Lower:    $309,725.27
  Upper:    $486,177.13

EQUIPMENT 2 - COST ESTIMATES

2025:
  Forecast: $217,713.17
  Lower:    $176,205.41
  Upper:    $259,228.16

2026:
  Forecast: $219,922.46
  Lower:    $177,107.49
  Upper:    $261,971.66

2027:
  Forecast: $222,129.61
  Lower:    $178,097.51
  Upper:    $265,532.22

3-YEAR TOTAL (2025-2027):
  Forecast: $659,765.24
  Lower:    $531,410.41
  Upper:    $786,732.04



El comportamiento presentado con el forecasting mensual muestra resultados mucho más confiables, evidenciado por un acotamiento significativo de la incertidumbre en el tiempo y un comportamiento acorde con las expectativas para la naturaleza de los datos. Los intervalos de confianza se mantienen en rangos razonables, permitiendo una planificación financiera más robusta.

Se presentan los valores estimados de los equipos para cada año (2025, 2026, 2027), considerando tres escenarios:
- **Escenario Base (Forecast)**: Predicción más probable basada en tendencias históricas
- **Escenario Optimista (Lower Bound)**: Límite inferior del intervalo de confianza, representa costos más bajos
- **Escenario Pesimista (Upper Bound)**: Límite superior del intervalo de confianza, representa costos más altos

Adicionalmente, se calcula el valor total de cada equipo para los 3 años del proyecto, proporcionando una visión consolidada del presupuesto requerido bajo cada escenario.